# Inference

Open this notebook with the **repository root** as the working directory, or run `pip install -e .` once from the repo root so `nmt` imports resolve from anywhere.

In [2]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "nmt").is_dir() and (_root.parent / "nmt").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)

import torch
from nmt.config import get_config, get_weights_path
from nmt.checkpoint import load_training_checkpoint
from nmt.train import get_model, get_dataset, run_validation
from nmt.translate import translate

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

model_filename = get_weights_path(config, str(config["checkpoint_epoch"]))
state = load_training_checkpoint(model_filename, map_location=device)
model.load_state_dict(state["model_state_dict"])

Using device: cpu


e:\transformer-translation-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\datasets--opus_books. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 51467/51467 [00:00<00:00, 462515.01 examples/s]


Max source length: 479, Max target length: 466


<All keys matched successfully>

In [4]:
run_validation(
    model,
    val_dataloader,
    tokenizer_src,
    tokenizer_tgt,
    config["seq_len"],
    device,
    lambda msg: print(msg),
    0,
    None,
    num_examples=5,
)

--------------------------------------------------------------------------------
    SOURCE: Diese hereingebrochene Dunkelheit gefiel ihm; er verlangte danach, eine dunkle und verlassene Gasse zu erreichen, wo er ruhig nachdenken könne, und damit der Philosoph auf die Wunde des Dichters den ersten Verband legen möchte.
    TARGET: This gloom pleased him; he was in haste to reach some obscure and deserted alley, in order there to meditate at his ease, and in order that the philosopher might place the first dressing upon the wound of the poet.
 PREDICTED: This gloom pleased him ; he demanded in a gloomy gloom and desert it , to reach time where he was quiet , and to enter the philosopher ' s wound upon the first wound of the poet .
--------------------------------------------------------------------------------
    SOURCE: Verdammt! Jetzt, wo wir Geld haben und 'ne Höhle und alles, was wir als Räuber brauchen, wirft einem so 'ne verrückte Tollheit alles übern Haufen!"
    TARGET: Blame i

In [5]:
t = translate("ein guter Student.")
print(f"Final translation: {t}")

Final translation: A good scholar .
